# HemoLens Hb Regression Model (v2 — optimized)

Input: the frozen 49 conjunctival color features produced by Notebook 1
(`eyelid_49_features.csv` + `feature_schema.json`).

Target: laboratory Hb in g/dL.

**What's new in this version compared to v1:**

- **Hyperparameter tuning** for every candidate model (Ridge, Random Forest,
  Extra Trees, and a new candidate, HistGradientBoosting) instead of fixed
  defaults, selected by cross-validation MAE.
- **Imbalance handling** for the small number of severe-anemia / low-Hb
  samples: SMOTE-style synthetic oversampling of minority Hb-severity bins
  in the *training* set only, plus inverse-frequency sample weights during
  fitting, so rare cases influence the model as much as common ones.
- **Leakage-safe out-of-fold calibration**: the out-of-fold error used to
  calibrate `hb_range` and `confidence` re-runs the oversampling *inside*
  each fold, so no synthetic sample derived from a validation-fold patient
  ever leaks into that fold's training data.
- **Better-calibrated confidence**: for tree-ensemble models, raw
  tree-to-tree disagreement is passed through an isotonic regression fitted
  against real out-of-fold error, so "confidence" tracks actual expected
  error rather than a heuristic ceiling.
- **Conditional save**: the new model is only written to disk if it beats
  the previously saved model on held-out test MAE.
- **New utility cells**: test any dataset image by `image_id`, test any
  arbitrary 49-feature vector, an accuracy-in-percentage summary, extra
  insights (feature importance, per-severity error), and a FastAPI-ready
  standalone inference module.

Final inference function's output (unchanged contract):
```
{
  "hb_estimate": 10.8,
  "hb_range": [9.75, 11.85],
  "confidence": 0.62
}
```

Output artifacts of this notebook:
- `eyelid_hb_model_v2.joblib`, saved to `My Drive/HemoLens/ML/models/` (only if it beats v1)
- `hemolens_predict.py`, a dependency-light module ready to drop into a FastAPI backend

This is a preliminary prototype and is **not** a clinically validated hemoglobin measurement system.

## Section 2 — Install dependencies

In [ ]:
!pip install -q numpy pandas scikit-learn scipy matplotlib joblib openpyxl

## Section 3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

RANDOM_STATE = 42

PROJECT_ROOT = "/content/drive/MyDrive/HemoLens"
OUTPUT_DIR = f"{PROJECT_ROOT}/ML"
PROCESSED_DIR = f"{OUTPUT_DIR}/processed"
MODELS_DIR = f"{OUTPUT_DIR}/models"

assert os.path.isdir(PROCESSED_DIR), f"Not found: {PROCESSED_DIR}. Did Notebook 1 finish?"
os.makedirs(MODELS_DIR, exist_ok=True)

print("Processed dir:", PROCESSED_DIR)
print("Models dir   :", MODELS_DIR)

## Section 4 — Load the 49-feature dataset

In [ ]:
import pandas as pd

print("Files in processed dir:")
for f in sorted(os.listdir(PROCESSED_DIR)):
    print(" -", f)

# EDIT this if the printed filename above differs.
FEATURE_CSV_NAME = "eyelid_49_features.csv"
FEATURE_FILE = f"{PROCESSED_DIR}/{FEATURE_CSV_NAME}"

assert os.path.isfile(FEATURE_FILE), (
    f"Not found: {FEATURE_FILE}. Check the filename against the list printed above "
    f"and update FEATURE_CSV_NAME."
)

df = pd.read_csv(FEATURE_FILE)
print("\nShape:", df.shape)
display(df.head())
print("\nMissing values per column:")
print(df.isna().sum())

## Section 5 — Verify the feature schema

We always build `X` by selecting columns by name in `FEATURE_NAMES` order,
never by raw column position — this is what protects you from training on
one feature order and predicting with another.

In [ ]:
import json

SCHEMA_FILE = f"{PROCESSED_DIR}/feature_schema.json"
assert os.path.isfile(SCHEMA_FILE), f"Not found: {SCHEMA_FILE}"

with open(SCHEMA_FILE) as f:
    schema = json.load(f)

FEATURE_NAMES = schema["feature_names"]
FEATURE_SCHEMA_VERSION = schema["feature_schema_version"]

assert len(FEATURE_NAMES) == 49, f"Expected 49 features in schema, got {len(FEATURE_NAMES)}"
missing_cols = [c for c in FEATURE_NAMES if c not in df.columns]
assert not missing_cols, f"CSV is missing expected feature columns: {missing_cols}"

print(f"Feature schema '{FEATURE_SCHEMA_VERSION}' verified: all {len(FEATURE_NAMES)} features present in CSV.")

## Section 6 — Inspect the target

`severity` / `remark` (if present) are shown only for context — they are
never used as model inputs. Note how few `Severe` and low-Hb cases exist;
Section 9 addresses this imbalance before any model sees the data.

In [ ]:
import matplotlib.pyplot as plt

print("Hb minimum :", df["hb"].min())
print("Hb maximum :", df["hb"].max())
print("Hb mean    :", df["hb"].mean())
print("Hb median  :", df["hb"].median())
print("Hb std     :", df["hb"].std())

plt.figure(figsize=(6, 4))
plt.hist(df["hb"].dropna(), bins=20)
plt.title("Hb distribution")
plt.xlabel("Hb (g/dL)")
plt.ylabel("Count")
plt.show()

if "severity" in df.columns:
    print("\nCounts by severity:")
    print(df["severity"].value_counts(dropna=False))

if "remark" in df.columns:
    print("\nCounts by remark:")
    print(df["remark"].value_counts(dropna=False))

## Section 7 — Define X and y

Strictly image-only model: `X` is exactly the 49 frozen features, `y` is lab
Hb. `age`, `gender`, `severity`, `remark`, `hospital`, `city`, etc. are never
included as inputs, even if present in the CSV.

In [ ]:
import numpy as np

X = df[FEATURE_NAMES].values.astype(np.float64)
y = df["hb"].values.astype(np.float64)

assert X.shape[1] == 49, f"Expected 49 feature columns, got {X.shape[1]}"
assert not np.isnan(X).any(), "X contains NaN -- go back and fix Notebook 1 output."
assert not np.isnan(y).any(), "y contains NaN -- rows with missing Hb should already be dropped."

print("X shape:", X.shape)
print("y shape:", y.shape)

## Section 8 — Train/test split

If a genuine patient identifier exists in the data, we use a group-aware
split so the same patient can't appear in both train and test. If not (as
with plain CP-AnemiC, where Notebook 1 found no such column), we document
that limitation and fall back to a standard random 80/20 split with a fixed
seed. `base_group_train` is defined either way — it is the grouping unit
every later cross-validation step (including synthetic-sample generation)
respects, so a real sample and anything derived from it always land in the
same fold.

In [ ]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit

HAS_PATIENT_ID = ("patient_id" in df.columns) and df["patient_id"].notna().all() and (df["patient_id"].nunique() > 1)

if HAS_PATIENT_ID:
    print("Patient IDs found -- using a group-aware split.")
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
    train_idx, test_idx = next(gss.split(X, y, groups=df["patient_id"]))
else:
    print("No usable patient ID column -- treating each image as an independent sample.")
    print("LIMITATION: if the same subject contributed more than one image, this split may leak information.")
    train_idx, test_idx = train_test_split(
        np.arange(len(df)), test_size=0.2, random_state=RANDOM_STATE
    )

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# The unit every group-aware CV split (including synthetic-sample folding)
# respects: the real patient id if we have one, otherwise each original row
# is treated as its own group.
if HAS_PATIENT_ID:
    base_group_train = df["patient_id"].values[train_idx]
else:
    base_group_train = train_idx.copy()

print("Train size:", len(X_train), " Test size:", len(X_test))

## Section 9 — Address data imbalance: synthetic oversampling + sample weights

The dataset has very few `Severe` and low-Hb cases (8 severe, 51 mild out of
413 total). Two complementary techniques compensate for this, applied to
the **training set only** — the test set is never touched, so test metrics
remain an honest measure of real-world performance:

1. **SMOTE-style oversampling for regression.** For each under-represented
   `severity` bin, new synthetic samples are generated by interpolating
   between a real sample and one of its nearest same-bin neighbors in
   feature space (with a little Gaussian jitter for diversity). The
   synthetic Hb target is interpolated the same way, so it stays consistent
   with the real local feature-target relationship rather than being
   invented from nothing. Each synthetic sample inherits its parent's group
   id so a group-aware CV split can never separate a synthetic sample from
   the real sample it came from.
2. **Inverse-frequency sample weights**, so during model fitting a rare
   `Severe` case counts for as much as several common `Non-Anemic` cases.

Both are re-applied *inside* the out-of-fold calibration loop later
(Section 15) using only that fold's own training portion, so the reported
out-of-fold and test metrics are not inflated by synthetic data leakage.

In [ ]:
from sklearn.neighbors import NearestNeighbors

severity_train = df["severity"].values[train_idx]
severity_train = np.array(["Unknown" if pd.isna(s) else s for s in severity_train])

print("Training severity distribution (before oversampling):")
print(pd.Series(severity_train).value_counts())


def smote_regression_oversample(X, y, bin_labels, groups, target_per_bin=None,
                                 k_neighbors=5, noise_scale=0.05, random_state=RANDOM_STATE):
    """SMOTE-style oversampling for a regression target.

    For each minority bin, synthesizes new samples by interpolating between
    a real sample and one of its k nearest same-bin neighbors in feature
    space, with small Gaussian jitter for extra diversity. The synthetic
    target value is interpolated the same way as the features, so it stays
    consistent with the local feature-target relationship. Each synthetic
    sample inherits its "parent" sample's group id, so a group-aware CV
    split never separates a synthetic sample from the real sample it was
    derived from.
    """
    rng = np.random.RandomState(random_state)
    bin_labels = np.asarray(bin_labels)
    unique_bins, counts = np.unique(bin_labels, return_counts=True)
    max_count = counts.max()
    if target_per_bin is None:
        target_per_bin = {b: max_count for b in unique_bins}

    X_syn_list, y_syn_list, groups_syn_list, bin_syn_list = [], [], [], []

    for b in unique_bins:
        idx_b = np.where(bin_labels == b)[0]
        n_have = len(idx_b)
        n_needed = max(0, int(target_per_bin.get(b, n_have)) - n_have)
        if n_needed == 0 or n_have < 2:
            continue
        X_b = X[idx_b]
        y_b = y[idx_b]
        k = min(k_neighbors, n_have - 1)
        nn = NearestNeighbors(n_neighbors=k + 1).fit(X_b)
        _, neighbor_idx = nn.kneighbors(X_b)

        for _ in range(n_needed):
            i = rng.randint(0, n_have)
            j = neighbor_idx[i, rng.randint(1, k + 1)]
            lam = rng.uniform(0.1, 0.9)
            x_new = X_b[i] + lam * (X_b[j] - X_b[i])
            y_new = y_b[i] + lam * (y_b[j] - y_b[i])
            feature_std = X_b.std(axis=0)
            x_new = x_new + rng.normal(0, noise_scale, size=x_new.shape) * feature_std
            X_syn_list.append(x_new)
            y_syn_list.append(y_new)
            groups_syn_list.append(groups[idx_b[i]])
            bin_syn_list.append(b)

    if not X_syn_list:
        return (np.empty((0, X.shape[1])), np.empty((0,)),
                np.empty((0,), dtype=groups.dtype), np.empty((0,), dtype=bin_labels.dtype))

    return (np.array(X_syn_list), np.array(y_syn_list),
            np.array(groups_syn_list), np.array(bin_syn_list))


X_syn, y_syn, groups_syn, severity_syn = smote_regression_oversample(
    X_train, y_train, severity_train, base_group_train
)

print(f"\nGenerated {len(X_syn)} synthetic samples to rebalance minority Hb-severity bins.")

X_train_aug = np.vstack([X_train, X_syn]) if len(X_syn) else X_train.copy()
y_train_aug = np.concatenate([y_train, y_syn]) if len(y_syn) else y_train.copy()
group_train_aug = np.concatenate([base_group_train, groups_syn]) if len(groups_syn) else base_group_train.copy()
severity_train_aug = np.concatenate([severity_train, severity_syn]) if len(severity_syn) else severity_train.copy()

print("\nAugmented training severity distribution:")
print(pd.Series(severity_train_aug).value_counts())
print("\nTrain size before augmentation:", len(X_train), " after augmentation:", len(X_train_aug))

In [ ]:
# Inverse-frequency sample weights (computed on the augmented training set)
# so common bins (e.g. Non-Anemic) don't dominate the loss and rare bins
# (e.g. Severe) get proportionally more influence during training.
bin_counts_aug = pd.Series(severity_train_aug).value_counts()
n_bins = len(bin_counts_aug)
n_total_aug = len(severity_train_aug)
bin_weight_map = {b: n_total_aug / (n_bins * c) for b, c in bin_counts_aug.items()}
sample_weight_aug = np.array([bin_weight_map[s] for s in severity_train_aug])
# Normalize so weights average to 1 (keeps loss magnitude comparable to unweighted)
sample_weight_aug = sample_weight_aug / sample_weight_aug.mean()

print("Per-bin weight multiplier:")
for b, w in bin_weight_map.items():
    print(f"  {b:12s}: {w:.3f}")

## Section 10 — Hyperparameter tuning

Instead of fixed defaults, each candidate model family is tuned over a
small randomized grid of hyperparameters, scored by cross-validation MAE on
the augmented, sample-weighted training set. `HistGradientBoosting` is
added as a fourth candidate alongside Ridge, Random Forest and Extra Trees.
Cross-validation uses `GroupKFold` on `group_train_aug`, so no fold ever
contains a synthetic sample without also containing the real sample it was
derived from.

Note: CV MAE at this tuning stage is used only to *compare and select*
models/hyperparameters — it can be mildly optimistic because augmentation
was done once on the whole training set rather than fold-by-fold. The
numbers reported later as genuine performance estimates (test-set metrics
in Section 13, and the fold-safe out-of-fold MAE in Section 15, which
re-runs augmentation inside each fold) are the ones to trust.

In [ ]:
import random
import itertools
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone
from scipy.stats import pearsonr

N_SPLITS = 5
MAX_COMBOS_PER_MODEL = 15  # raise this for a more thorough (slower) search

cv_splitter = GroupKFold(n_splits=N_SPLITS)
cv_splits = list(cv_splitter.split(X_train_aug, y_train_aug, groups=group_train_aug))


def make_model(name, params):
    if name == "Ridge":
        return Pipeline([("scaler", StandardScaler()), ("model", Ridge(random_state=RANDOM_STATE, **params))])
    if name == "Random Forest":
        return RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
    if name == "Extra Trees":
        return ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
    if name == "HistGradientBoosting":
        return HistGradientBoostingRegressor(random_state=RANDOM_STATE, **params)
    raise ValueError(f"Unknown model name: {name}")


def evaluate_fold(model, X_tr, y_tr, w_tr, X_val, y_val, w_val):
    m = clone(model)
    if isinstance(m, Pipeline):
        m.fit(X_tr, y_tr, model__sample_weight=w_tr)
    else:
        m.fit(X_tr, y_tr, sample_weight=w_tr)
    pred = m.predict(X_val)
    mae = mean_absolute_error(y_val, pred, sample_weight=w_val)
    rmse = np.sqrt(mean_squared_error(y_val, pred, sample_weight=w_val))
    r2 = r2_score(y_val, pred, sample_weight=w_val)
    pearson = pearsonr(y_val, pred)[0] if len(np.unique(pred)) > 1 else np.nan
    return mae, rmse, r2, pearson


def cv_score_params(model_ctor, splits):
    fold_metrics = []
    for tr, val in splits:
        model = model_ctor()
        fold_metrics.append(evaluate_fold(
            model,
            X_train_aug[tr], y_train_aug[tr], sample_weight_aug[tr],
            X_train_aug[val], y_train_aug[val], sample_weight_aug[val],
        ))
    fold_metrics = np.array(fold_metrics)
    return {
        "mae_mean": fold_metrics[:, 0].mean(),
        "mae_std": fold_metrics[:, 0].std(),
        "rmse_mean": fold_metrics[:, 1].mean(),
        "r2_mean": fold_metrics[:, 2].mean(),
        "pearson_mean": np.nanmean(fold_metrics[:, 3]),
    }


param_grids = {
    "Ridge": [{"alpha": a} for a in [0.1, 1.0, 5.0, 10.0, 30.0, 100.0]],
    "Random Forest": [
        {"n_estimators": n, "max_depth": d, "min_samples_leaf": l, "max_features": mf}
        for n in [300, 500]
        for d in [None, 8, 12]
        for l in [1, 2, 4]
        for mf in ["sqrt", 0.5]
    ],
    "Extra Trees": [
        {"n_estimators": n, "max_depth": d, "min_samples_leaf": l, "max_features": mf}
        for n in [300, 500]
        for d in [None, 8, 12]
        for l in [1, 2, 4]
        for mf in ["sqrt", 0.5]
    ],
    "HistGradientBoosting": [
        {"max_iter": it, "max_leaf_nodes": ln, "learning_rate": lr, "min_samples_leaf": l, "l2_regularization": l2}
        for it in [200, 400]
        for ln in [15, 31]
        for lr in [0.03, 0.06, 0.1]
        for l in [10, 20]
        for l2 in [0.0, 1.0]
    ],
}

random.seed(RANDOM_STATE)


def sample_combos(grid, max_combos):
    if len(grid) <= max_combos:
        return grid
    return random.sample(grid, max_combos)


tuning_results = []
best_per_model = {}

for name, grid in param_grids.items():
    combos = sample_combos(grid, MAX_COMBOS_PER_MODEL)
    print(f"Tuning {name}: evaluating {len(combos)} candidate configurations...")
    best_score = None
    for params in combos:
        model_ctor = lambda p=params, nm=name: make_model(nm, p)
        metrics = cv_score_params(model_ctor, cv_splits)
        tuning_results.append({"model": name, "params": params, **metrics})
        if best_score is None or metrics["mae_mean"] < best_score["mae_mean"]:
            best_score = metrics
            best_per_model[name] = {"params": params, **metrics}
    print(f"  best {name} CV MAE: {best_per_model[name]['mae_mean']:.3f}  (params: {best_per_model[name]['params']})")

tuning_results_df = pd.DataFrame(tuning_results).sort_values("mae_mean").reset_index(drop=True)
display(tuning_results_df.head(15))

## Section 11 — Model comparison and selection

Selection is based on the lowest tuned cross-validation MAE. The data
decides — not an assumption about which model "should" work best.

In [ ]:
model_comparison_df = pd.DataFrame([
    {"model": name, **info} for name, info in best_per_model.items()
]).sort_values("mae_mean").reset_index(drop=True)

display(model_comparison_df)

best_model_name = model_comparison_df.iloc[0]["model"]
best_params = model_comparison_df.iloc[0]["params"]
print("Selected model (lowest tuned CV MAE):", best_model_name)
print("Selected hyperparameters:", best_params)

## Section 12 — Train the selected model

Fit a fresh (unfit) copy of the chosen, tuned model on the entire
augmented, sample-weighted training set.

In [ ]:
final_model = make_model(best_model_name, best_params)

if isinstance(final_model, Pipeline):
    final_model.fit(X_train_aug, y_train_aug, model__sample_weight=sample_weight_aug)
else:
    final_model.fit(X_train_aug, y_train_aug, sample_weight=sample_weight_aug)

print(f"{best_model_name} trained on {len(X_train_aug)} samples "
      f"({len(X_train)} real + {len(X_syn)} synthetic), using inverse-frequency sample weights.")

## Section 13 — Final test evaluation

The held-out test set was never augmented, reweighted, or otherwise
touched, so these numbers are an honest measure of performance on real
images only.

In [ ]:
y_pred_test = final_model.predict(X_test)

mae_test = mean_absolute_error(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2_test = r2_score(y_test, y_pred_test)
pearson_test = pearsonr(y_test, y_pred_test)[0]

print(f"Test MAE     : {mae_test:.3f}")
print(f"Test RMSE    : {rmse_test:.3f}")
print(f"Test R^2     : {r2_test:.3f}")
print(f"Test Pearson : {pearson_test:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(y_test, y_pred_test, alpha=0.6)
lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
axes[0].plot(lims, lims, "r--")
axes[0].set_xlabel("Actual Hb")
axes[0].set_ylabel("Predicted Hb")
axes[0].set_title("Actual vs Predicted")

residuals = y_test - y_pred_test
axes[1].scatter(y_pred_test, residuals, alpha=0.6)
axes[1].axhline(0, color="r", linestyle="--")
axes[1].set_xlabel("Predicted Hb")
axes[1].set_ylabel("Actual - Predicted")
axes[1].set_title("Residuals")

axes[2].hist(np.abs(residuals), bins=15)
axes[2].set_xlabel("Absolute error (g/dL)")
axes[2].set_ylabel("Count")
axes[2].set_title("Absolute error distribution")

plt.tight_layout()
plt.show()

## Section 14 — Helper: per-sample ensemble disagreement

For Random Forest / Extra Trees, each tree in the ensemble gives its own
prediction for a given input; the spread (standard deviation) across trees
is a genuine per-sample uncertainty signal. This helper is reused for the
out-of-fold calibration below and inside the final, FastAPI-ready inference
function.

In [ ]:
def get_tree_std(model, X_rows):
    """Return per-row std across an ensemble's trees (RF/Extra Trees), or
    None if the model doesn't expose one (Ridge, HistGradientBoosting)."""
    inner = model.named_steps["model"] if isinstance(model, Pipeline) else model
    estimators = getattr(inner, "estimators_", None)
    if estimators is None:
        return None
    scaler = model.named_steps.get("scaler") if isinstance(model, Pipeline) else None
    X_use = scaler.transform(X_rows) if scaler is not None else X_rows
    per_tree = np.array([t.predict(X_use) for t in estimators])
    return per_tree.std(axis=0)

## Section 15 — Out-of-fold predictions (for calibration)

For every training sample, get a prediction from a model that never saw
that sample during its own fold's fit. Unlike Section 10's tuning search,
this loop re-runs the SMOTE oversampling and sample weighting *inside* each
fold's own training portion only, so these out-of-fold residuals are a
leakage-safe estimate of real-world error. They are what we calibrate
`hb_range` and `confidence` on next — using the test set for this would
quietly leak information into the calibration.

In [ ]:
raw_signal_oof = np.full(len(X_train), np.nan)
oof_pred = np.full(len(X_train), np.nan)

oof_cv = GroupKFold(n_splits=N_SPLITS)
for tr_idx, val_idx in oof_cv.split(X_train, y_train, groups=base_group_train):
    X_tr_fold, y_tr_fold = X_train[tr_idx], y_train[tr_idx]
    sev_tr_fold = severity_train[tr_idx]
    grp_tr_fold = base_group_train[tr_idx]

    X_syn_f, y_syn_f, _, sev_syn_f = smote_regression_oversample(
        X_tr_fold, y_tr_fold, sev_tr_fold, grp_tr_fold
    )
    X_tr_fold_aug = np.vstack([X_tr_fold, X_syn_f]) if len(X_syn_f) else X_tr_fold
    y_tr_fold_aug = np.concatenate([y_tr_fold, y_syn_f]) if len(y_syn_f) else y_tr_fold
    sev_tr_fold_aug = np.concatenate([sev_tr_fold, sev_syn_f]) if len(sev_syn_f) else sev_tr_fold

    bin_counts_f = pd.Series(sev_tr_fold_aug).value_counts()
    w_map_f = {b: len(sev_tr_fold_aug) / (len(bin_counts_f) * c) for b, c in bin_counts_f.items()}
    w_tr_fold_aug = np.array([w_map_f[s] for s in sev_tr_fold_aug])
    w_tr_fold_aug = w_tr_fold_aug / w_tr_fold_aug.mean()

    fold_model = make_model(best_model_name, best_params)
    if isinstance(fold_model, Pipeline):
        fold_model.fit(X_tr_fold_aug, y_tr_fold_aug, model__sample_weight=w_tr_fold_aug)
    else:
        fold_model.fit(X_tr_fold_aug, y_tr_fold_aug, sample_weight=w_tr_fold_aug)

    oof_pred[val_idx] = fold_model.predict(X_train[val_idx])
    raw = get_tree_std(fold_model, X_train[val_idx])
    if raw is not None:
        raw_signal_oof[val_idx] = raw

assert not np.isnan(oof_pred).any(), "Some samples never appeared in a validation fold."

oof_residuals = y_train - oof_pred
oof_abs_residuals = np.abs(oof_residuals)
print("Out-of-fold MAE:", oof_abs_residuals.mean())

plt.figure(figsize=(6, 4))
plt.hist(oof_abs_residuals, bins=20)
plt.title("Out-of-fold absolute error distribution")
plt.xlabel("Absolute error (g/dL)")
plt.ylabel("Count")
plt.show()

## Section 16 — Calibrate the Hb range

The margin is read from the validation data, not hard-coded. Change
`CALIBRATION_PERCENTILE` if you want a wider or narrower band.

In [ ]:
CALIBRATION_PERCENTILE = 90
hb_margin = float(np.percentile(oof_abs_residuals, CALIBRATION_PERCENTILE))
print(f"{CALIBRATION_PERCENTILE}th percentile out-of-fold absolute error: {hb_margin:.3f} g/dL")
print("hb_range for any prediction p will be [p - hb_margin, p + hb_margin].")

## Section 17 — Define model confidence

This is a model confidence score — never call it, or present it as, a
probability of anemia or a probability of correct diagnosis.

For Random Forest / Extra Trees, raw tree-to-tree disagreement (from
Section 14's helper) is passed through an **isotonic regression** fitted
against the real out-of-fold absolute error, so the "expected error" behind
each confidence score is empirically grounded rather than an arbitrary
scale. Ridge and HistGradientBoosting have no per-sample ensemble signal,
so they fall back to the global calibration margin from Section 16 as their
expected error for every prediction.

Confidence is then `1 - (expected_error / error_ceiling)`, clipped to
`[0, 1]`, where `error_ceiling` is a high percentile of the out-of-fold
error distribution (an error this large or larger is treated as ~0
confidence).

In [ ]:
from sklearn.isotonic import IsotonicRegression

USES_TREE_ENSEMBLE = best_model_name in ("Random Forest", "Extra Trees")

if USES_TREE_ENSEMBLE and not np.isnan(raw_signal_oof).any():
    error_calibrator = IsotonicRegression(out_of_bounds="clip", increasing=True)
    error_calibrator.fit(raw_signal_oof, oof_abs_residuals)
    print("Fitted an isotonic calibrator mapping tree-ensemble disagreement -> expected absolute error.")
else:
    error_calibrator = None
    print(f"Selected model ('{best_model_name}') has no usable per-sample ensemble-disagreement "
          "signal; confidence will fall back to the global calibration margin for every prediction.")

ERROR_CEILING_PERCENTILE = 99
ERROR_CEILING = float(np.percentile(oof_abs_residuals, ERROR_CEILING_PERCENTILE))
print("Error ceiling (treated as ~0 confidence):", ERROR_CEILING)

## Section 18 — Export the final model (only if it improves on the previous one)

The saved bundle carries more than the three output fields (model version,
calibration parameters, test metrics) because a FastAPI backend needs that
for versioning and debugging — `predict_hb_from_bundle`'s return value to
the API is still just `hb_estimate`, `hb_range`, `confidence`. The new
model is only written to disk if its test MAE beats the previously saved
model's test MAE, so a bad tuning run can never silently overwrite a good
model.

In [ ]:
import joblib
from datetime import datetime, timezone

MODEL_VERSION = "eyelid_hb_model_v2"
MODEL_PATH = f"{MODELS_DIR}/eyelid_hb_model_v2.joblib"
PREVIOUS_MODEL_PATH = f"{MODELS_DIR}/eyelid_hb_model_v1.joblib"

candidate_bundle = {
    "model": final_model,
    "model_name": best_model_name,
    "model_params": best_params,
    "feature_names": FEATURE_NAMES,
    "feature_schema_version": FEATURE_SCHEMA_VERSION,
    "calibration_margin": hb_margin,
    "calibration_percentile": CALIBRATION_PERCENTILE,
    "error_ceiling": ERROR_CEILING,
    "error_ceiling_percentile": ERROR_CEILING_PERCENTILE,
    "uses_tree_ensemble_confidence": USES_TREE_ENSEMBLE,
    "error_calibrator": error_calibrator,
    "model_version": MODEL_VERSION,
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "oof_mae": float(oof_abs_residuals.mean()),
    "test_metrics": {
        "mae": mae_test,
        "rmse": rmse_test,
        "r2": r2_test,
        "pearson": pearson_test,
    },
    "n_train_real": len(X_train),
    "n_train_synthetic": len(X_syn),
}

previous_test_mae = None
if os.path.isfile(PREVIOUS_MODEL_PATH):
    try:
        prev_bundle = joblib.load(PREVIOUS_MODEL_PATH)
        previous_test_mae = prev_bundle.get("test_metrics", {}).get("mae")
    except Exception as e:
        print("Could not load previous model bundle for comparison:", e)

print("New model test MAE     :", round(mae_test, 3))
print("Previous model test MAE:", previous_test_mae)

model_improved = (previous_test_mae is None) or (mae_test < previous_test_mae)

if model_improved:
    joblib.dump(candidate_bundle, MODEL_PATH)
    print("New model is better (or no previous model found) -- saved to:", MODEL_PATH)
    MODEL_SAVED_PATH = MODEL_PATH
else:
    print("New model did NOT improve on the previous saved model -- not overwriting.")
    print(f"Kept previous model at: {PREVIOUS_MODEL_PATH}")
    MODEL_SAVED_PATH = PREVIOUS_MODEL_PATH

print("\nModel in use for the rest of this notebook:", MODEL_SAVED_PATH)

## Section 19 — Reload, verify, and define the FastAPI-ready prediction function

Reload the `.joblib` file from disk (as if this were a fresh process, like
your FastAPI server starting up) and confirm it produces a working
prediction end-to-end. `predict_hb_from_bundle` is a **pure function** — it
takes only a feature vector and a loaded bundle, uses no notebook globals,
and can be copied as-is into a FastAPI service (Section 24 writes it to its
own file for exactly that purpose).

In [ ]:
def _vector_from_input(feature_input, feature_names):
    if isinstance(feature_input, dict):
        missing = [f for f in feature_names if f not in feature_input]
        if missing:
            raise ValueError(f"Missing features in input: {missing}")
        vec = np.array([feature_input[f] for f in feature_names], dtype=np.float64)
    else:
        vec = np.asarray(feature_input, dtype=np.float64)
    if vec.shape[0] != len(feature_names):
        raise ValueError(f"Expected {len(feature_names)} features, got {vec.shape[0]}")
    if not np.all(np.isfinite(vec)):
        raise ValueError("Feature vector contains NaN or Inf values.")
    return vec


def predict_hb_from_bundle(feature_input, bundle):
    """Pure function: takes a raw feature vector/dict and a loaded model
    bundle, returns exactly the three fields the API contract promises."""
    vec = _vector_from_input(feature_input, bundle["feature_names"])
    model = bundle["model"]
    hb_estimate = float(model.predict(vec.reshape(1, -1))[0])

    expected_error = bundle["calibration_margin"]
    if bundle.get("uses_tree_ensemble_confidence") and bundle.get("error_calibrator") is not None:
        raw = get_tree_std(model, vec.reshape(1, -1))
        if raw is not None:
            expected_error = float(bundle["error_calibrator"].predict(raw)[0])

    expected_error = max(expected_error, 0.0)
    error_ceiling = bundle["error_ceiling"]
    confidence = 1.0 - (expected_error / error_ceiling) if error_ceiling > 0 else 1.0
    confidence = float(np.clip(confidence, 0.0, 1.0))

    margin = bundle["calibration_margin"]
    return {
        "hb_estimate": round(hb_estimate, 2),
        "hb_range": [round(hb_estimate - margin, 2), round(hb_estimate + margin, 2)],
        "confidence": round(confidence, 2),
    }


loaded_bundle = joblib.load(MODEL_SAVED_PATH)
assert loaded_bundle["feature_names"] == FEATURE_NAMES, "Feature order mismatch after reload!"
print("Reloaded model bundle from:", MODEL_SAVED_PATH)
print("Model:", loaded_bundle["model_name"], "| version:", loaded_bundle["model_version"])

for idx in test_idx[:5]:
    row = df.iloc[idx]
    feature_vec = row[FEATURE_NAMES].to_dict()
    result = predict_hb_from_bundle(feature_vec, loaded_bundle)
    print(f"image_id={row['image_id']} actual_hb={row['hb']:.2f} -> {result}")

## Section 20 — Test by `image_id`

Look up any image already in the dataset by its `image_id` (e.g.
`"Image_001"`), run it through the saved model, and compare the prediction
against the lab-measured Hb.

In [ ]:
def test_by_image_id(image_id, bundle=loaded_bundle, dataframe=df):
    matches = dataframe[dataframe["image_id"] == image_id]
    if matches.empty:
        raise ValueError(f"image_id '{image_id}' not found in the dataset.")
    row = matches.iloc[0]
    feature_vec = row[FEATURE_NAMES].to_dict()
    result = predict_hb_from_bundle(feature_vec, bundle)
    actual_hb = float(row["hb"])
    within_range = result["hb_range"][0] <= actual_hb <= result["hb_range"][1]

    print(f"image_id        : {image_id}")
    print(f"dataset split   : {'train' if row.name in train_idx else 'test'}")
    print(f"severity/remark : {row.get('severity')} / {row.get('remark')}")
    print(f"feature vector  : {feature_vec}")
    print(f"predicted hb    : {result['hb_estimate']} g/dL")
    print(f"predicted range : {result['hb_range']} g/dL")
    print(f"confidence      : {result['confidence']}")
    print(f"actual hb       : {actual_hb} g/dL")
    print(f"within range?   : {'YES' if within_range else 'no'}")
    return result


# EDIT this to test any image_id in the dataset
test_by_image_id("Image_001")

## Section 21 — Test by a raw feature vector

Edit `custom_feature_vector` below with your own 49 feature values (a dict
keyed by feature name, or a plain list/array of 49 numbers already in
`FEATURE_NAMES` order both work) to test the model on values that didn't
come from the dataset at all — e.g. straight from your own image-processing
pipeline.

In [ ]:
# EDIT this dict with your own 49 feature values.
custom_feature_vector = {name: 0.0 for name in FEATURE_NAMES}

# Quick way to start from a real row instead of typing all 49 values by hand:
# custom_feature_vector = df.iloc[0][FEATURE_NAMES].to_dict()
# custom_feature_vector["r_mean"] = 50.0

try:
    result = predict_hb_from_bundle(custom_feature_vector, loaded_bundle)
    print("Prediction for custom feature vector:")
    print(result)
except ValueError as e:
    print("Fix your input and re-run:", e)

## Section 22 — Model accuracy, in percentage

Regression models don't have a single native "accuracy %", so three
complementary percentages are reported:

1. **Range-coverage accuracy** — the % of test images whose actual Hb falls
   inside the model's own predicted `hb_range`. This is the most directly
   useful number for judging whether the reported range can be trusted.
2. **Point-accuracy proxy** — `100 * (1 - MAE / mean(actual Hb))`, an
   intuitive "how close on average" percentage. It is *not* a standard
   regression metric, just an easy-to-read summary.
3. **Clinical anemic / non-anemic classification accuracy** — thresholding
   both actual and predicted Hb at 11 g/dL and scoring it like a binary
   classifier, since flagging likely anemia is often the more clinically
   useful downstream task than the exact Hb number.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# 1) Range-coverage accuracy
test_hb_range_lo = y_pred_test - hb_margin
test_hb_range_hi = y_pred_test + hb_margin
within_range = (y_test >= test_hb_range_lo) & (y_test <= test_hb_range_hi)
range_coverage_pct = 100.0 * within_range.mean()

# 2) Point-accuracy proxy (intuitive, not a formal metric)
point_accuracy_pct = 100.0 * (1.0 - mae_test / y_test.mean())

# 3) Clinical anemic/non-anemic classification accuracy
ANEMIA_THRESHOLD = 11.0
actual_anemic = y_test < ANEMIA_THRESHOLD
predicted_anemic = y_pred_test < ANEMIA_THRESHOLD

clinical_accuracy_pct = 100.0 * accuracy_score(actual_anemic, predicted_anemic)
clinical_precision = precision_score(actual_anemic, predicted_anemic, zero_division=0)
clinical_recall = recall_score(actual_anemic, predicted_anemic, zero_division=0)
clinical_f1 = f1_score(actual_anemic, predicted_anemic, zero_division=0)
cm = confusion_matrix(actual_anemic, predicted_anemic)

print(f"Range-coverage accuracy (actual Hb within predicted range) : {range_coverage_pct:.1f}%")
print(f"Point-accuracy proxy (1 - MAE/mean(actual Hb))             : {point_accuracy_pct:.1f}%")
print("  (this is an intuitive summary number, not a standard regression metric)")
print()
print(f"Clinical anemic/non-anemic classification accuracy (threshold={ANEMIA_THRESHOLD} g/dL):")
print(f"  Accuracy  : {clinical_accuracy_pct:.1f}%")
print(f"  Precision : {clinical_precision:.3f}  (of predicted-anemic, how many really are)")
print(f"  Recall    : {clinical_recall:.3f}  (of truly anemic, how many were caught)")
print(f"  F1        : {clinical_f1:.3f}")
print("  Confusion matrix [rows=actual, cols=predicted], order=[non-anemic, anemic]:")
print(cm)

## Section 23 — Additional insights

In [ ]:
# Feature importance (tree-based models only)
inner_model = final_model.named_steps["model"] if isinstance(final_model, Pipeline) else final_model
importances = getattr(inner_model, "feature_importances_", None)

if importances is not None:
    importance_df = pd.DataFrame({"feature": FEATURE_NAMES, "importance": importances}) \
        .sort_values("importance", ascending=False).reset_index(drop=True)
    print("Top 10 most important features:")
    display(importance_df.head(10))
    plt.figure(figsize=(8, 5))
    plt.barh(importance_df["feature"].head(10)[::-1], importance_df["importance"].head(10)[::-1])
    plt.xlabel("Importance")
    plt.title("Top 10 feature importances")
    plt.tight_layout()
    plt.show()
else:
    print(f"{best_model_name} does not expose feature_importances_.")

# Error by severity subgroup on the test set
severity_test = df["severity"].values[test_idx]
error_by_severity = pd.DataFrame({
    "severity": severity_test,
    "abs_error": np.abs(y_test - y_pred_test),
}).groupby("severity")["abs_error"].agg(["mean", "count"]).sort_values("mean")
print("\nMean absolute error by severity group (test set):")
display(error_by_severity)

severe_mask = severity_test == "Severe"
if severe_mask.sum() > 0:
    print(f"\nSevere-anemia test subset (n={int(severe_mask.sum())}): "
          f"MAE={np.abs(y_test[severe_mask] - y_pred_test[severe_mask]).mean():.2f} g/dL. "
          "Interpret with caution -- this subgroup is very small.")
else:
    print("\nNo 'Severe' cases landed in the test split (small original subgroup, n=8 overall) -- "
          "MAE for severe anemia specifically could not be measured on this split.")

print(f"\nSynthetic oversampling added {len(X_syn)} samples to under-represented Hb-severity "
      "bins in the training set (Section 9); this does not manufacture test data, so the "
      "test metrics above remain an honest measure of real-world performance on genuine images only.")

## Section 24 — FastAPI integration prep

Writes a dependency-light, self-contained `hemolens_predict.py` module next
to the saved model — it only needs `numpy`, `joblib`, and `scikit-learn`
(for unpickling the `Pipeline` / `IsotonicRegression` objects stored inside
the bundle) at inference time, and has no dependency on this notebook.
Input: a 49-feature vector. Output: `hb_estimate`, `hb_range`, `confidence`.

In [ ]:
fastapi_module_code = '''"""
hemolens_predict.py

Drop this file into your FastAPI backend. It has no dependency on the
training notebook -- only numpy, joblib and scikit-learn (for the Pipeline /
IsotonicRegression classes stored inside the bundle) are required at
inference time.

Usage in FastAPI:

    from hemolens_predict import load_bundle, predict_hb_from_bundle

    bundle = load_bundle("eyelid_hb_model_v2.joblib")

    @app.post("/predict")
    def predict(features: FeatureVector):
        return predict_hb_from_bundle(features.dict(), bundle)
"""
import numpy as np
import joblib
from sklearn.pipeline import Pipeline


def load_bundle(model_path):
    return joblib.load(model_path)


def _vector_from_input(feature_input, feature_names):
    if isinstance(feature_input, dict):
        missing = [f for f in feature_names if f not in feature_input]
        if missing:
            raise ValueError(f"Missing features in input: {missing}")
        vec = np.array([feature_input[f] for f in feature_names], dtype=np.float64)
    else:
        vec = np.asarray(feature_input, dtype=np.float64)
    if vec.shape[0] != len(feature_names):
        raise ValueError(f"Expected {len(feature_names)} features, got {vec.shape[0]}")
    if not np.all(np.isfinite(vec)):
        raise ValueError("Feature vector contains NaN or Inf values.")
    return vec


def get_tree_std(model, X_rows):
    inner = model.named_steps["model"] if isinstance(model, Pipeline) else model
    estimators = getattr(inner, "estimators_", None)
    if estimators is None:
        return None
    scaler = model.named_steps.get("scaler") if isinstance(model, Pipeline) else None
    X_use = scaler.transform(X_rows) if scaler is not None else X_rows
    per_tree = np.array([t.predict(X_use) for t in estimators])
    return per_tree.std(axis=0)


def predict_hb_from_bundle(feature_input, bundle):
    """Returns exactly {"hb_estimate", "hb_range", "confidence"}."""
    vec = _vector_from_input(feature_input, bundle["feature_names"])
    model = bundle["model"]
    hb_estimate = float(model.predict(vec.reshape(1, -1))[0])

    expected_error = bundle["calibration_margin"]
    if bundle.get("uses_tree_ensemble_confidence") and bundle.get("error_calibrator") is not None:
        raw = get_tree_std(model, vec.reshape(1, -1))
        if raw is not None:
            expected_error = float(bundle["error_calibrator"].predict(raw)[0])

    expected_error = max(expected_error, 0.0)
    error_ceiling = bundle["error_ceiling"]
    confidence = 1.0 - (expected_error / error_ceiling) if error_ceiling > 0 else 1.0
    confidence = float(np.clip(confidence, 0.0, 1.0))

    margin = bundle["calibration_margin"]
    return {
        "hb_estimate": round(hb_estimate, 2),
        "hb_range": [round(hb_estimate - margin, 2), round(hb_estimate + margin, 2)],
        "confidence": round(confidence, 2),
    }
'''

FASTAPI_MODULE_PATH = f"{MODELS_DIR}/hemolens_predict.py"
with open(FASTAPI_MODULE_PATH, "w") as f:
    f.write(fastapi_module_code)

print("Wrote FastAPI-ready inference module to:", FASTAPI_MODULE_PATH)
print("\nMinimal FastAPI app skeleton (for reference -- lives in your API repo, not here):")
print('''
from fastapi import FastAPI
from pydantic import BaseModel, create_model
from hemolens_predict import load_bundle, predict_hb_from_bundle

BUNDLE = load_bundle("eyelid_hb_model_v2.joblib")
FeatureVector = create_model(
    "FeatureVector", **{name: (float, ...) for name in BUNDLE["feature_names"]}
)

app = FastAPI()

@app.post("/predict")
def predict(features: FeatureVector):
    return predict_hb_from_bundle(features.dict(), BUNDLE)
''')

## Done — what changed and what to check

- [ ] Hyperparameter tuning ran for all four candidate models (Section 10-11)
- [ ] Synthetic oversampling + sample weighting rebalanced the Severe/low-Hb
      minority (Section 9), and the leakage-safe OOF loop (Section 15)
      re-applied it per-fold rather than reusing the single global augmentation
- [ ] Confidence is isotonic-calibrated against real out-of-fold error where
      a tree ensemble was selected (Section 17)
- [ ] The new model was only saved if it beat the previous saved model on
      test MAE (Section 18) — check the printed comparison
- [ ] `test_by_image_id(...)` and the custom-feature-vector cell both work
      (Sections 20-21)
- [ ] Accuracy-in-percentage and insights cells ran without error (Sections 22-23)
- [ ] `hemolens_predict.py` was written to `MODELS_DIR` and is ready to copy
      into the FastAPI backend (Section 24)

This remains a preliminary prototype and is **not** a clinically validated
hemoglobin measurement system.